<a href="https://colab.research.google.com/github/24msrdf014/My-Page/blob/main/CyberRangeActivity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
import random
import threading
import time
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime

from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout,
    QLabel, QPushButton, QTableWidget, QTableWidgetItem,
    QTextEdit, QProgressBar
)
from PyQt5.QtCore import Qt

import matplotlib.pyplot as plt

# =========================
# DATABASE SETUP
# =========================
conn = sqlite3.connect("insider_threat.db", check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS logs (
    user_id TEXT,
    timestamp TEXT,
    action TEXT,
    location TEXT,
    bytes_transferred INTEGER
)
""")
conn.commit()

# =========================
# GLOBAL VARIABLES
# =========================
users = ["U101", "U102", "U103"]
risk_scores = {u: 0 for u in users}
locked_users = set()
activity_data = []

# =========================
# MAIN WINDOW
# =========================
class ThreatSimulator(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Insider Threat Simulator - PyQt Dashboard")
        self.setGeometry(100, 100, 900, 600)

        self.layout = QVBoxLayout()

        # Top Info
        self.info_label = QLabel("🧑‍💻 Insider Threat Monitoring Dashboard")
        self.info_label.setAlignment(Qt.AlignCenter)
        self.layout.addWidget(self.info_label)

        # Table
        self.table = QTableWidget()
        self.table.setColumnCount(5)
        self.table.setHorizontalHeaderLabels(
            ["User", "Time", "Action", "Location", "Bytes"]
        )
        self.layout.addWidget(self.table)

        # Risk Bars
        self.risk_layout = QHBoxLayout()
        self.risk_bars = {}

        for user in users:
            bar = QProgressBar()
            bar.setMaximum(10)
            bar.setFormat(f"{user} Risk: %p%")
            self.risk_bars[user] = bar
            self.risk_layout.addWidget(bar)

        self.layout.addLayout(self.risk_layout)

        # Alerts Box
        self.alert_box = QTextEdit()
        self.alert_box.setReadOnly(True)
        self.layout.addWidget(self.alert_box)

        # Buttons
        self.btn_layout = QHBoxLayout()

        self.graph_btn = QPushButton("Show Activity Graph")
        self.graph_btn.clicked.connect(self.show_graph)
        self.btn_layout.addWidget(self.graph_btn)

        self.layout.addLayout(self.btn_layout)

        self.setLayout(self.layout)

        # Start threads
        threading.Thread(target=self.simulate_activity, daemon=True).start()
        threading.Thread(target=self.update_ui, daemon=True).start()

    # =========================
    # SIMULATION
    # =========================
    def simulate_activity(self):
        while True:
            user = random.choice(users)

            if user in locked_users:
                continue

            now = datetime.now()

            if random.random() < 0.8:
                action = "file_access"
                location = "public"
                bytes_tx = random.randint(100, 1000)
            else:
                action = random.choice(["bulk_download", "restricted_access"])
                location = "restricted"
                bytes_tx = random.randint(5000, 20000)

            log = {
                "user_id": user,
                "timestamp": now.strftime("%H:%M:%S"),
                "action": action,
                "location": location,
                "bytes_transferred": bytes_tx
            }

            self.insert_log(log)
            activity_data.append(log)

            self.detect_anomaly(user, log)

            time.sleep(1)

    def insert_log(self, log):
        cursor.execute("INSERT INTO logs VALUES (?, ?, ?, ?, ?)",
                       (log["user_id"], log["timestamp"],
                        log["action"], log["location"],
                        log["bytes_transferred"]))
        conn.commit()

    # =========================
    # DETECTION
    # =========================
    def detect_anomaly(self, user, log):
        df = pd.DataFrame(activity_data)

        if len(df) < 10:
            return

        user_df = df[df["user_id"] == user]

        if len(user_df) < 5:
            return

        mean = user_df["bytes_transferred"].mean()
        std = user_df["bytes_transferred"].std()

        if std == 0:
            return

        z = (log["bytes_transferred"] - mean) / std

        if z > 2:
            risk_scores[user] += 2
            self.add_alert(f"[ALERT] {user} anomaly detected! Z={round(z,2)}")

        if log["location"] == "restricted":
            risk_scores[user] += 3

        if log["action"] == "bulk_download":
            risk_scores[user] += 3

        if risk_scores[user] > 8:
            locked_users.add(user)
            self.add_alert(f"[DEFENSE] {user} ACCOUNT LOCKED!")

    # =========================
    # UI UPDATE
    # =========================
    def update_ui(self):
        while True:
            if activity_data:
                log = activity_data[-1]
                row = self.table.rowCount()
                self.table.insertRow(row)

                self.table.setItem(row, 0, QTableWidgetItem(log["user_id"]))
                self.table.setItem(row, 1, QTableWidgetItem(log["timestamp"]))
                self.table.setItem(row, 2, QTableWidgetItem(log["action"]))
                self.table.setItem(row, 3, QTableWidgetItem(log["location"]))
                self.table.setItem(row, 4, QTableWidgetItem(str(log["bytes_transferred"])))

            for user in users:
                self.risk_bars[user].setValue(min(risk_scores[user], 10))

            time.sleep(1)

    def add_alert(self, msg):
        self.alert_box.append(msg)

    # =========================
    # GRAPH
    # =========================
    def show_graph(self):
        df = pd.DataFrame(activity_data)

        if df.empty:
            return

        df.groupby("user_id")["bytes_transferred"].sum().plot(kind="bar")
        plt.title("User Activity Volume")
        plt.show()


# =========================
# RUN
# =========================
if __name__ == "__main__":
    app = QApplication(sys.argv)
    window = ThreatSimulator()
    window.show()
    sys.exit(app.exec_())

ModuleNotFoundError: No module named 'PyQt5'